# MedGemma read service (kaggle)
GPU runtime on. Needs the HF_TOKEN secret (account must have accepted the model terms).

In [ ]:
import os, subprocess
print(subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout)
!nvidia-smi -L

In [ ]:
# Store HF_TOKEN as a secret first (never paste it in a cell).
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['MEDGEMMA_QUANT'] = 'nf4'
os.environ['MEDGEMMA_COMPUTE'] = 'fp16'  # T4/P100 have no bf16; try 'fp32' if output is garbage

In [ ]:
!pip -q install 'transformers>=4.50' accelerate bitsandbytes fastapi uvicorn python-multipart pillow pydantic

In [ ]:
!git clone https://github.com/<ORG>/<REPO>.git repo && cd repo && git log -1 --oneline

In [ ]:
%cd repo
!python -m services.medgemma.smoke --image services/medgemma/fixtures/smoke_cxr.png

## Serve + tunnel
Start the server, then expose it with a cloudflared quick tunnel (no account). Put the printed `https://*.trycloudflare.com` URL in `MEDGEMMA_URL` and in progress.md (P3 section).

In [ ]:
import subprocess, time
subprocess.Popen(['uvicorn','services.medgemma.server:app','--port','8001'])
time.sleep(5)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8001'])

## Batch reads
`python -m services.medgemma.batch --images <dir> --modality cxr --out ml/artifacts/medgemma_reads/<dataset>.jsonl` (resumable; re-run after a disconnect).